<a href="https://colab.research.google.com/github/RachelAlcraft/TrivialDivergence/blob/main/Triangles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ra-trivial

In [102]:
# Settings
import random
import math
import pandas as pd

#columns =['A', 'B','C','AB','BC','CA','CBp']
#columns =['A', 'B','C']
columns =['A', 'B','C','BC','AB','CA']
bins = 10
no_tri= 10
max_dims = len(columns)
dic_lists = {}

In [103]:
# triangle making function
def make_triangles(no_triangles,right_all_mix):

  # R=Right, A=any, M e half right half any for 2 states
  bond_length_angle_range = (45,55)
  bond_length_range_lower = (5,15)
  bond_length_range_mid = (5,15)
  bond_length_range_upper = (5,15)
  AB_range = [5,8]

  As = []
  Bs = []
  Cs = []
  ABs = []
  BCs = []
  CAs = []
  CBps = []

  for i in range(no_tri):
    this_r_a_m = right_all_mix
    if right_all_mix == "M":
      if i < no_tri/2:
        this_r_a_m = "R"
      else:
        this_r_a_m = "A"
    if this_r_a_m == "R":
      A = 90
    elif i == 0:
      A = 90 #I want to make sure I capture the random possibility of a different state so always have at least 1 right angle triangle
    else:
      A = random.randint(1, 178)
    B = random.randint(1, 180-A-1)
    C = 180 - (A+B)
    Ar = math.radians(A)
    Br = math.radians(B)
    Cr = math.radians(C)
    AB = random.randint(5,8)
    sin_ratio = AB / math.sin(Cr)
    BC = math.sin(Ar) * sin_ratio
    CA = math.sin(Br) * sin_ratio
    #CBp is the link to the next triangle
    if C < bond_length_angle_range[0]:
      CBp = random.randint(bond_length_range_lower[0],bond_length_range_lower[1])
    elif C > bond_length_angle_range[1]:
      CBp = random.randint(bond_length_range_upper[0],bond_length_range_upper[1])
    else:
      CBp = random.randint(bond_length_range_mid[0],bond_length_range_mid[1])
    #print(A,B,C,A+B+C,AB,BC,CA,CBp)
    As.append(A)
    Bs.append(B)
    Cs.append(C)
    ABs.append(AB)
    BCs.append(BC)
    CAs.append(CA)
    CBps.append(CBp)
  ####################################

  dic_lists["A"] = As
  dic_lists["B"] = Bs
  dic_lists["C"] = Cs
  dic_lists["AB"] = ABs
  dic_lists["BC"] = BCs
  dic_lists["CA"] = CAs
  dic_lists["CBp"] = CBps

  df_list = []
  for col in columns:
    df_list.append( dic_lists[col])

  df_transpose =  [list(i) for i in zip(*df_list)]
  df = pd.DataFrame(df_transpose,columns =columns)
  return df

In [105]:
# generate a triangle dataset

dfR = make_triangles(no_tri,"R")
dfA = make_triangles(no_tri,"A")
dfM = make_triangles(no_tri,"M")
#print(dfR)

dic_dfs_per_tri = {}
dic_dfs_per_tri["Mixed"] = dfM
dic_dfs_per_tri["Right"] = dfR
dic_dfs_per_tri["Any"] = dfA

#print(dfM)
#print(dfR)
#print(dfA)

print("Completed")

Completed


In [106]:
# Sampling each observation FUNCTION
# We now want to try again removing each obs and recalcing
def drop_row(df,cols,bins,dim,df_corr,show=False):
  list_x = df_corr["metric"].tolist()
  col_names = []
  for i in range(len(df_corr.index)):
    cols_nm = ""
    for c in range(dim):
      col_nm = f"col{c+1}"
      cols_nm += ":" + df_corr[col_nm][i]
    col_names.append(cols_nm[1:])
  if show:
    print(col_names)

  if show:
    print("x",list_x)
  # list of lists object for heatmap
  list_of_lists = []
  list_of_metrics = []
  for i in range(len(df.index)):
    dfx=df.drop(df.index[i])
    rae_markx = awa.AlcraftWilliamsAssociation(dfx,bins=bins,piters=0)
    df_corrx = rae_markx.getStrongestAssociations([],cols,dim,1)
    list_i = df_corrx["metric"].tolist()
    df_corrx['is_it_bigger'] = df_corrx['metric'] > (df_corr['metric'])
    #df_corrx["is_it_bigger"] = df_corrx["is_it_bigger"].astype(str) #make it a string
    list_TF = df_corrx["is_it_bigger"].tolist()
    list_met = df_corrx["metric"].tolist()
    #print(i,list_i)
    if show:
      print(i,list_TF)
    list_of_lists.append(list_TF)
    list_of_metrics.append(list_met)
  return col_names,list_of_lists,list_of_metrics


In [107]:
# Create the trivial divergence
cols = columns
import ra_trivial.AlcraftWilliamsAssociation as awa

sets_corrs = {}

for tag,df in dic_dfs_per_tri.items():
#for tag,df in [("Mixed",dfM)]:
  rae_mark = awa.AlcraftWilliamsAssociation(df,bins=bins,piters=0)
  for i in range(2,max_dims+1):
    print(tag,"\tmake",i,"dims")
    df_corri = rae_mark.getStrongestAssociations([],cols,i,1)
    sets_corrs[f"{tag}_{i}"] = df_corri


print("Complete")

Mixed 	make 2 dims
Mixed 	make 3 dims
Mixed 	make 4 dims
Mixed 	make 5 dims
Mixed 	make 6 dims
Right 	make 2 dims
Right 	make 3 dims
Right 	make 4 dims
Right 	make 5 dims
Right 	make 6 dims
Any 	make 2 dims
Any 	make 3 dims
Any 	make 4 dims
Any 	make 5 dims
Any 	make 6 dims
Complete


In [108]:
# make into a heatmap
import plotly.express as px
# https://plotly.com/python/heatmaps/

def make_heatmap(list_of_lists, col_names, title,hue):

  fig = px.imshow(list_of_lists,
                  labels=dict(x="Association", y="Observations?", color=hue),
                  x=col_names,
                  color_continuous_scale='Bluered_r',
                  title=title)
  fig.update_xaxes(side="top")
  fig.show()



In [109]:
# Create the plots and individual samples

for tag,df in dic_dfs_per_tri.items():
  print(tag)
  show_rows = False
  all_col_names = []
  dic_lists = {}
  dic_metrics = {}
  print("Drop rows")
  for i in range(2,max_dims+1):
    print(f"{i}/{max_dims}")
    dfi = sets_corrs[f"{tag}_{i}"]
    col_namesi,list_of_listsi,list_of_metricsi = drop_row(df,cols,bins,i,dfi,show=show_rows)
    dic_lists[i] = list_of_listsi
    dic_metrics[i] = list_of_metricsi
    all_col_names += col_namesi

  list_of_list_of_lists = []
  list_of_list_of_metrics = []
  num_obs = len(dic_lists[2]) # get the length of the first one, it is the number of observations
  print("Create obs lists")
  for ob in range(num_obs):
    lolol = []
    for key,lol in dic_lists.items():
      obs_lst = lol[ob]
      lolol += obs_lst
    list_of_list_of_lists.append(lolol)
  for ob in range(num_obs):
    lolol = []
    for key,lol in dic_metrics.items():
      obs_lst = lol[ob]
      lolol += obs_lst
    list_of_list_of_metrics.append(lolol)
  # Make HEATMAP
  print("number obs=",num_obs,tag)
  make_heatmap(list_of_list_of_lists, all_col_names, title=f"{tag} Improves by removal?",hue="Improves?")
  make_heatmap(list_of_list_of_metrics, all_col_names,title=f"{tag} Metric with removal",hue="Triviality")

Mixed
Drop rows
2/6
3/6
4/6
5/6
6/6
Create obs lists
number obs= 10 Mixed


Right
Drop rows
2/6
3/6
4/6
5/6
6/6
Create obs lists
number obs= 10 Right


Any
Drop rows
2/6
3/6
4/6
5/6
6/6
Create obs lists
number obs= 10 Any


In [ ]:
# build a tree with all dims at the top and then each with a thing removed
# and the 10101 for each of those